<a href="https://colab.research.google.com/github/fre-caneo/Cienciadedatos/blob/Sumativa1/S2_Entrega1_Cuaderno_Caneo_Freddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrega 1 · Adquisición, calidad y preparación de datos

## Siniestros de tránsito geocodificados · Chile, 2021

**Pregunta analítica:** En los registros de siniestros geocodificados de CONASET para 2021, ¿cómo varía la frecuencia por tramo horario y qué tipos y causas aparecen con mayor frecuencia en el tramo con más registros?

Este cuaderno trabaja con **57.902 registros** de la capa publicada por CONASET. Cada fila corresponde a un siniestro de esa capa. La selección incluye `AÑO = 2021` y `Zonas_urb_ = Urbano`; el campo original `ZONA` contiene también 1.363 registros clasificados como rurales. Por eso los resultados describen la capa seleccionada y no el total nacional.


## 0. Fuente y configuración

- **Ficha del conjunto:** [Siniestros individuales 2021 · CONASET](https://mapas-conaset.opendata.arcgis.com/datasets/a1d136edbdde4252904241a5386df86d_0)
- **Fuente original declarada por CONASET:** registros de siniestros de Carabineros de Chile.
- **Capa consultada:** `Base_SINIESTROS_2020_2025`, filtrada por `AÑO = 2021` y `Zonas_urb_ = 'Urbano'`.
- **Extracción usada en esta entrega:** `S2_Siniestros_CONASET_2021_Urbano_ORIGINAL.csv`.
- **Fecha de descarga:** 25 de octubre de 2026.
- **Versión:** la ficha no publica un número formal; se identifica esta copia como la extracción del 25 de octubre de 2026 con el filtro indicado.
- **Licencia:** la ficha del elemento de ArcGIS consultada no declara una licencia de reutilización; cita a CONASET y confirma las condiciones de uso antes de redistribuir.
- **Repositorio:** [Cienciadedatos · rama Sumativa1](https://github.com/fre-caneo/Cienciadedatos/tree/Sumativa1). La rama todavía tiene una versión anterior del cuaderno; antes de entregar, sube este cuaderno corregido y el conjunto limpio versionado.

En Google Colab, sube el CSV original cuando la celda siguiente lo solicite. Junto a este cuaderno se entrega una copia descargada para que el análisis use la misma extracción.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

ARCHIVO = Path('S2_Siniestros_CONASET_2021_Urbano_ORIGINAL.csv')
if not ARCHIVO.exists():
    try:
        from google.colab import files
        cargados = files.upload()
        if not cargados:
            raise FileNotFoundError('No se seleccionó el CSV original.')
        ARCHIVO = Path(next(iter(cargados)))
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            'Deja el CSV original junto al cuaderno o súbelo desde Google Colab.'
        ) from exc

df_original = pd.read_csv(ARCHIVO, encoding='utf-8-sig')
print('Archivo:', ARCHIVO.name)
print('Filas y columnas originales:', df_original.shape)


Archivo: S2_Siniestros_CONASET_2021_Urbano_ORIGINAL.csv
Filas y columnas originales: (57902, 36)


## 1. Cargar y conocer los datos

`df_original` conserva las 36 columnas entregadas por la capa, antes de limpiar. Revisa algunas variables de identificación, tiempo, clasificación y resultados.


In [2]:
print('Columnas originales:')
for col in df_original.columns:
    print('-', repr(col))
print('\nMuestra de variables analíticas:')
display_cols = ['ID', 'AÑO', 'FECHA', 'Dia_semana', 'Hora', 'Hora_aprox',
                'REGION', 'COMUNA', 'TIPO__CONA', 'CAUSA_NUEV',
                'FALLECIDOS', 'TOTAL_LESI']
print(df_original[display_cols].head(8).to_string(index=False))


Columnas originales:
- 'FID'
- 'ID'
- 'AÑO'
- 'FECHA'
- 'Dia_semana'
- 'Mes'
- 'Hora'
- 'Hora_aprox'
- 'REGION'
- 'PROVINCIA'
- 'COMUNA'
- 'TIPO_SINIE'
- 'TIPO__CONA'
- 'ZONA'
- 'CAUSA_SINI'
- 'CAUSA_NUEV'
- 'UBICACION'
- 'Calle_Uno'
- 'Calle_Dos'
- 'Número'
- 'Ruta'
- 'Ubicación'
- 'Dirección'
- 'FALLECIDOS'
- 'TOTAL_LESI'
- 'GRAVES'
- 'MENOS_GRAV'
- 'LEVES'
- 'Siniestro'
- 'Latitud'
- 'longitud'
- 'Zonas_urb_'
- 'Atropello'
- 'Bicicleta'
- 'Motociclet'
- 'AM'

Muestra de variables analíticas:
     ID  AÑO         FECHA Dia_semana    Hora  Hora_aprox                                REGION          COMUNA TIPO__CONA                CAUSA_NUEV  FALLECIDOS  TOTAL_LESI
1324060 2021 1609459200000    Viernes 0:05:00           0                          La Araucanía           Purén     CHOQUE      ALCOHOL EN CONDUCTOR           0           1
1327715 2021 1609459200000    Viernes 0:20:00           0                            Valparaíso      Valparaíso     CHOQUE              OTRAS CAUSAS      

## 2. Pregunta, unidad de análisis y trazabilidad

**Pregunta analítica:** En los registros de siniestros geocodificados de CONASET para 2021, ¿cómo varía la frecuencia por tramo horario y qué tipos y causas aparecen con mayor frecuencia en el tramo con más registros?

**Unidad de análisis:** una fila representa un registro de siniestro en la capa de CONASET seleccionada. `Hora_aprox` permite agrupar por hora; `TIPO__CONA` y `CAUSA_NUEV` describen tipo y causa; `REGION`, `COMUNA`, `ZONA` y los resultados de personas permiten contextualizarlo.

La ficha corresponde a una capa geocodificada. Se usa el filtro publicado para 2021 (`AÑO = 2021` y `Zonas_urb_ = Urbano`), que devuelve 57.902 filas. El campo `ZONA` mantiene la clasificación original del registro: 56.539 Urbana y 1.363 Rural. No se interpreta `Zonas_urb_` como sustituto de `ZONA`.


In [3]:
print('Años:', df_original['AÑO'].value_counts(dropna=False).to_dict())
print('Clasificación de la capa geocodificada:', df_original['Zonas_urb_'].value_counts(dropna=False).to_dict())
print('Zona declarada en el registro:', df_original['ZONA'].value_counts(dropna=False).to_dict())
print('Registros:', len(df_original))


Años: {2021: 57902}
Clasificación de la capa geocodificada: {'Urbano': 57902}
Zona declarada en el registro: {'URBANA': 56539, 'RURAL': 1363}
Registros: 57902


## 3. Diagnóstico de calidad antes de limpiar

La completitud se calcula tratando como faltante tanto el valor nulo como las celdas de texto compuestas solo por espacios. Se conserva `df_original` sin modificar para que el diagnóstico sea reproducible.


In [4]:
df_evaluacion = df_original.copy()
for col in df_evaluacion.select_dtypes(include=['object', 'string']).columns:
    df_evaluacion[col] = df_evaluacion[col].replace(r'^\s*$', np.nan, regex=True)
faltantes = df_evaluacion.isna().sum().sort_values(ascending=False)
print('Columnas con faltantes o texto vacío:')
print(faltantes[faltantes.gt(0)].to_string())
print('\nPorcentaje de celdas vacías en el archivo:',
      round(df_evaluacion.isna().sum().sum() / df_evaluacion.size * 100, 2), '%')


/tmp/ipykernel_515/771799569.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_evaluacion[col] = df_evaluacion[col].replace(r'^\s*$', np.nan, regex=True)


Columnas con faltantes o texto vacío:
Ruta         57902
AM           20174
Calle_Dos    17870

Porcentaje de celdas vacías en el archivo: 4.6 %


### Patrón de faltantes y revisión de posibles centinelas

Los faltantes de `AM` y `Calle_Dos` tienen mayor proporción en `ZONA = RURAL`; `Ruta` está vacía en ambos grupos. Como la ficha no entrega un diccionario de códigos centinela, se revisan códigos numéricos y marcadores textuales habituales. Esta búsqueda no descarta códigos institucionales no documentados. El cero se conserva como conteo válido de personas fallecidas o lesionadas.


In [5]:
campos_faltantes = ['AM', 'Calle_Dos', 'Ruta']
print('Faltantes por ZONA (conteo y porcentaje dentro de cada grupo):')
for zona, grupo in df_evaluacion.groupby('ZONA', dropna=False):
    resultados = []
    for columna in campos_faltantes:
        n = int(grupo[columna].isna().sum())
        pct = round(n / len(grupo) * 100, 1) if len(grupo) else 0
        resultados.append(f'{columna}: {n}/{len(grupo)} ({pct}%)')
    print(f'{zona}: ' + ' | '.join(resultados))


Faltantes por ZONA (conteo y porcentaje dentro de cada grupo):
RURAL: AM: 935/1363 (68.6%) | Calle_Dos: 966/1363 (70.9%) | Ruta: 1363/1363 (100.0%)
URBANA: AM: 19239/56539 (34.0%) | Calle_Dos: 16904/56539 (29.9%) | Ruta: 56539/56539 (100.0%)


In [6]:
columnas_conteo_hora = ['Hora_aprox', 'FALLECIDOS', 'TOTAL_LESI', 'GRAVES', 'MENOS_GRAV', 'LEVES']
candidatos_numericos = [-1, 99, 999]
print('Candidatos numéricos -1, 99, 999 en hora/conteos:')
for columna in columnas_conteo_hora:
    encontrados = {v: int(df_original[columna].eq(v).sum()) for v in candidatos_numericos
                   if df_original[columna].eq(v).any()}
    print(f'{columna}: {encontrados if encontrados else "ninguno"}')

marcadores_textuales = {'N/A', 'NA', 'N/D', 'S/D', 'SIN DATO', 'NO INFORMADO', 'DESCONOCIDO', 'UNKNOWN'}
hallazgos_textuales = {}
for columna in df_original.select_dtypes(include=['object', 'string']).columns:
    normalizada = df_original[columna].astype('string').str.strip().str.upper()
    encontrados = normalizada[normalizada.isin(marcadores_textuales)].value_counts()
    if len(encontrados):
        hallazgos_textuales[columna] = encontrados.to_dict()
print('Marcadores textuales candidatos:', hallazgos_textuales if hallazgos_textuales else 'ninguno')
print('Cero se conserva como conteo válido; no se interpreta como faltante.')


Candidatos numéricos -1, 99, 999 en hora/conteos:
Hora_aprox: ninguno
FALLECIDOS: ninguno
TOTAL_LESI: ninguno
GRAVES: ninguno
MENOS_GRAV: ninguno
LEVES: ninguno
Marcadores textuales candidatos: ninguno
Cero se conserva como conteo válido; no se interpreta como faltante.


In [7]:
COLUMNAS_RESULTADO = ['FALLECIDOS', 'TOTAL_LESI', 'GRAVES', 'MENOS_GRAV', 'LEVES']
conteos = df_original[COLUMNAS_RESULTADO]
print('Rangos de resultados por registro:')
print(conteos.agg(['min', 'max']).to_string())
print('\n¿Hay valores negativos?', bool(conteos.lt(0).any().any()))
print('¿Hay conteos no enteros?', bool((conteos % 1 != 0).any().any()))
print('Rango Hora_aprox:', int(df_original['Hora_aprox'].min()), 'a', int(df_original['Hora_aprox'].max()))
print('Eventos con fallecidos:', int(df_original['FALLECIDOS'].gt(0).sum()))
print('Eventos con lesionados:', int(df_original['TOTAL_LESI'].gt(0).sum()))


Rangos de resultados por registro:
     FALLECIDOS  TOTAL_LESI  GRAVES  MENOS_GRAV  LEVES
min           0           0       0           0      0
max           3          20       5           7     14

¿Hay valores negativos? False
¿Hay conteos no enteros? False
Rango Hora_aprox: 0 a 23
Eventos con fallecidos: 495
Eventos con lesionados: 22130


In [8]:
fecha_dt = pd.to_datetime(df_original['FECHA'], unit='ms', errors='coerce')
hora_dt = pd.to_datetime(df_original['Hora'], format='%H:%M:%S', errors='coerce')
anio_fecha = fecha_dt.dt.year
hora_coincide = hora_dt.dt.hour.eq(df_original['Hora_aprox'])
suma_lesiones = (df_original['GRAVES'] + df_original['MENOS_GRAV'] + df_original['LEVES'])
reglas = {
    'FECHA coincide con AÑO = 2021': int(anio_fecha.eq(df_original['AÑO']).sum()),
    'Hora_aprox coincide con Hora': int(hora_coincide.sum()),
    'TOTAL_LESI coincide con la suma de severidades': int(df_original['TOTAL_LESI'].eq(suma_lesiones).sum()),
}
for regla, casos in reglas.items():
    print(f'{regla}: {casos}/{len(df_original)}')
print('\nRango de fecha:', fecha_dt.min().date(), 'a', fecha_dt.max().date())
print('Horas inválidas:', int(hora_dt.isna().sum()))


FECHA coincide con AÑO = 2021: 57902/57902
Hora_aprox coincide con Hora: 57902/57902
TOTAL_LESI coincide con la suma de severidades: 57902/57902

Rango de fecha: 2021-01-01 a 2021-12-31
Horas inválidas: 0


In [9]:
print('Duplicados de fila completa:', int(df_original.duplicated().sum()))
print('ID únicos:', int(df_original['ID'].nunique()), 'de', len(df_original))
print('FID únicos:', int(df_original['FID'].nunique()), 'de', len(df_original))


Duplicados de fila completa: 0
ID únicos: 57902 de 57902
FID únicos: 57902 de 57902


### Resumen de calidad

La variable `AM` está vacía en 20.174 filas; `Calle_Dos` contiene 17.870 celdas solo con espacios y `Ruta` está vacía en las 57.902 filas. Esos campos no se usan para responder la pregunta. Las variables analíticas de fecha, hora, tipo, causa, comuna y resultados no presentan faltantes.


In [10]:
resumen_calidad = pd.DataFrame([
    ['Completitud', 'Sin faltantes analíticos. AM: 68,6% rural vs 34,0% urbana; Calle_Dos: 70,9% vs 29,9%; Ruta: 100% en ambos grupos.'],
    ['Exactitud', 'Hora válida 00–23; conteos no negativos; máximo 3 fallecidos y 20 lesionados. No se hallan candidatos centinela habituales; cero es válido.'],
    ['Consistencia', 'FECHA/AÑO: 57.902 de 57.902; Hora/Hora_aprox: 57.902 de 57.902; suma de lesiones: 57.902 de 57.902.'],
    ['Unicidad', '0 filas duplicadas; ID y FID únicos en los 57.902 registros.'],
    ['Oportunidad', 'Periodo 2021-01-01 a 2021-12-31; capa histórica para el análisis del año observado.'],
], columns=['Dimensión', 'Resultado antes de limpiar'])
print(resumen_calidad.to_string(index=False))


   Dimensión                                                                                                                  Resultado antes de limpiar
 Completitud                           Sin faltantes analíticos. AM: 68,6% rural vs 34,0% urbana; Calle_Dos: 70,9% vs 29,9%; Ruta: 100% en ambos grupos.
   Exactitud Hora válida 00–23; conteos no negativos; máximo 3 fallecidos y 20 lesionados. No se hallan candidatos centinela habituales; cero es válido.
Consistencia                                         FECHA/AÑO: 57.902 de 57.902; Hora/Hora_aprox: 57.902 de 57.902; suma de lesiones: 57.902 de 57.902.
    Unicidad                                                                                0 filas duplicadas; ID y FID únicos en los 57.902 registros.
 Oportunidad                                                         Periodo 2021-01-01 a 2021-12-31; capa histórica para el análisis del año observado.


## 4. Bitácora de decisiones y tratamiento de valores extremos

Las decisiones se documentan en la bitácora Excel adjunta. Se mantienen las filas: el trabajo reduce columnas, convierte la fecha y construye un tramo horario; no imputa valores ni elimina registros de la capa.

- **Cobertura:** usar el filtro de la ficha para 2021 y la capa geocodificada; limitar las conclusiones a esa selección. `ZONA` conserva 1.363 registros etiquetados como rurales.
- **Fecha:** convertir `FECHA` desde milisegundos Unix a una fecha legible; los 57.902 registros corresponden al año 2021.
- **Columnas:** conservar ID, fecha, hora, territorio, tipo, causa y resultados. Excluir direcciones exactas, coordenadas, columnas auxiliares y campos incompletos que no responden la pregunta.
- **Valores extremos:** conservar los máximos de 3 fallecidos y 20 lesionados por registro. No son evidencia suficiente de error y todos los totales de lesiones cuadran con sus severidades.


In [11]:
COLUMNAS_ANALISIS = [
    'ID', 'AÑO', 'FECHA', 'Dia_semana', 'Mes', 'Hora', 'Hora_aprox',
    'REGION', 'PROVINCIA', 'COMUNA', 'TIPO_SINIE', 'TIPO__CONA',
    'CAUSA_SINI', 'CAUSA_NUEV', 'ZONA', 'Zonas_urb_',
    'FALLECIDOS', 'TOTAL_LESI', 'GRAVES', 'MENOS_GRAV', 'LEVES'
]
TRAMOS = ['Madrugada (00–05)', 'Mañana (06–11)', 'Tarde (12–17)', 'Noche (18–23)']

def limpiar(df):
    d = df.copy()
    d.columns = [str(col).strip() for col in d.columns]
    filas_entrada = len(d)
    d = d.loc[d['AÑO'].eq(2021) & d['Zonas_urb_'].astype('string').str.strip().eq('Urbano')].copy()
    d['FECHA'] = pd.to_datetime(d['FECHA'], unit='ms', errors='coerce')
    d = d[COLUMNAS_ANALISIS].copy()
    d['Tramo_horario'] = pd.cut(
        d['Hora_aprox'], bins=[-1, 5, 11, 17, 23], labels=TRAMOS, right=True
    )
    informe = {
        'Filas al inicio': filas_entrada,
        'Filas dentro del alcance 2021/capa urbana': len(d),
        'Filas descartadas por filtro': filas_entrada - len(d),
        'Columnas analíticas conservadas': len(COLUMNAS_ANALISIS),
        'Atributos derivados': ['Tramo_horario'],
    }
    return d.reset_index(drop=True), informe


### Ejecutar la limpieza y comprobar reproducibilidad

La función conserva los 21 campos analíticos, convierte `FECHA` y crea `Tramo_horario`. Los tramos tienen seis horas cada uno: madrugada 00–05, mañana 06–11, tarde 12–17 y noche 18–23.


In [12]:
df_limpio, informe_limpieza = limpiar(df_original)
print(informe_limpieza)
print('Filas/columnas limpias:', df_limpio.shape)
print(df_limpio.head(8).to_string(index=False))


{'Filas al inicio': 57902, 'Filas dentro del alcance 2021/capa urbana': 57902, 'Filas descartadas por filtro': 0, 'Columnas analíticas conservadas': 21, 'Atributos derivados': ['Tramo_horario']}
Filas/columnas limpias: (57902, 22)
     ID  AÑO      FECHA Dia_semana   Mes    Hora  Hora_aprox                                REGION  PROVINCIA          COMUNA             TIPO_SINIE TIPO__CONA                                        CAUSA_SINI                CAUSA_NUEV   ZONA Zonas_urb_  FALLECIDOS  TOTAL_LESI  GRAVES  MENOS_GRAV  LEVES     Tramo_horario
1324060 2021 2021-01-01    Viernes Enero 0:05:00           0                          La Araucanía    Malleco           Purén         CHOQUE FRONTAL     CHOQUE                  CONDUCCION EN ESTADO DE EBRIEDAD      ALCOHOL EN CONDUCTOR URBANA     Urbano           0           1       0           0      1 Madrugada (00–05)
1327715 2021 2021-01-01    Viernes Enero 0:20:00           0                            Valparaíso Valparaíso      Valparaí

In [13]:
a, _ = limpiar(df_original)
b, _ = limpiar(df_original)
print('¿limpiar() entrega el mismo resultado dos veces?', a.equals(b))


¿limpiar() entrega el mismo resultado dos veces? True


### Valores extremos

La variable `TOTAL_LESI` alcanza 20 personas en un registro y `FALLECIDOS` alcanza 3. Se conservan porque no son imposibles por sí mismos, no hay conteos negativos y la suma de lesiones graves, menos graves y leves coincide con `TOTAL_LESI` en todas las filas. Esta coherencia interna no reemplaza la validación con el parte original de Carabineros.


In [14]:
print('Máximo fallecidos por registro:', int(df_limpio['FALLECIDOS'].max()))
print('Máximo lesionados por registro:', int(df_limpio['TOTAL_LESI'].max()))
print('Total fallecidos en la capa:', int(df_limpio['FALLECIDOS'].sum()))
print('Total lesionados en la capa:', int(df_limpio['TOTAL_LESI'].sum()))
print('Ningún registro eliminado:', len(df_limpio) == len(df_original))


Máximo fallecidos por registro: 3
Máximo lesionados por registro: 20
Total fallecidos en la capa: 514
Total lesionados en la capa: 30221
Ningún registro eliminado: True


## 5. Atributo construido y análisis descriptivo

`Tramo_horario` agrupa `Hora_aprox` en cuatro intervalos iguales de seis horas. Permite comparar dónde se concentra el mayor número de registros y revisar qué tipos y causas aparecen más en ese intervalo. Son **frecuencias dentro de la capa**, no tasas de riesgo: no tenemos flujo vehicular ni exposición por hora.


In [15]:
frecuencia_tramo = df_limpio['Tramo_horario'].value_counts(sort=False).rename('Siniestros').to_frame()
frecuencia_tramo['Porcentaje'] = (frecuencia_tramo['Siniestros'] / len(df_limpio) * 100).round(2)
print(frecuencia_tramo.to_string())
print('\nHora con más registros:')
por_hora = df_limpio['Hora_aprox'].value_counts().sort_index()
hora_pico = int(por_hora.idxmax())
print(f'{hora_pico:02d}:00 — {int(por_hora.max()):,} registros ({por_hora.max()/len(df_limpio)*100:.2f}%)')


                   Siniestros  Porcentaje
Tramo_horario                            
Madrugada (00–05)        3969        6.85
Mañana (06–11)          14412       24.89
Tarde (12–17)           21493       37.12
Noche (18–23)           18028       31.14

Hora con más registros:
18:00 — 4,497 registros (7.77%)


In [16]:
tramo_pico = frecuencia_tramo['Siniestros'].idxmax()
df_pico = df_limpio.loc[df_limpio['Tramo_horario'].eq(tramo_pico)]
print('Tramo con más registros:', tramo_pico, '—', len(df_pico))
print('\nTipos más frecuentes en el tramo:')
print(df_pico['TIPO__CONA'].value_counts().head(5).to_string())
print('\nCausas más frecuentes en el tramo:')
print(df_pico['CAUSA_NUEV'].value_counts().head(6).to_string())


Tramo con más registros: Tarde (12–17) — 21493

Tipos más frecuentes en el tramo:
TIPO__CONA
COLISION     13020
CHOQUE        6028
ATROPELLO     1508
VOLCADURA      493
OTRO TIPO      289

Causas más frecuentes en el tramo:
CAUSA_NUEV
DISTRACCION DEL CONDUCTOR       8443
IMPRUDENCIA DEL CONDUCTOR       4755
DESOBEDIENCIA A SEÑALIZACION    2763
CAUSAS NO DETERMINADAS          1634
OTRAS CAUSAS                    1332
VELOCIDAD IMPRUDENTE            1105


In [17]:
print('Distribución por hora (cada bloque representa aproximadamente el 2,5 % del máximo):')
maximo_hora = int(por_hora.max())
for hora, cantidad in por_hora.items():
    barra = '█' * round(cantidad / maximo_hora * 40)
    print(f'{hora:02d}:00 | {barra:<40} {cantidad:>5,}')


Distribución por hora (cada bloque representa aproximadamente el 2,5 % del máximo):
00:00 | █████████                                  999
01:00 | ███████                                    772
02:00 | ██████                                     625
03:00 | █████                                      548
04:00 | ████                                       482
05:00 | █████                                      543
06:00 | ████████                                   874
07:00 | ███████████████████                      2,188
08:00 | ███████████████████████████              3,014
09:00 | ███████████████████████                  2,576
10:00 | ████████████████████████                 2,728
11:00 | ███████████████████████████              3,032
12:00 | ███████████████████████████████          3,462
13:00 | █████████████████████████████████        3,742
14:00 | █████████████████████████████            3,232
15:00 | ████████████████████████████████         3,545
16:00 | ████████████████████████████

In [18]:
tipos_pico = df_pico['TIPO__CONA'].value_counts().sort_values(ascending=False)
print(f'Tipos de siniestro en {tramo_pico}:')
maximo_tipo = int(tipos_pico.max())
for tipo, cantidad in tipos_pico.items():
    barra = '█' * round(cantidad / maximo_tipo * 40)
    print(f'{tipo:<15} | {barra:<40} {cantidad:>5,}')


Tipos de siniestro en Tarde (12–17):
COLISION        | ████████████████████████████████████████ 13,020
CHOQUE          | ███████████████████                      6,028
ATROPELLO       | █████                                    1,508
VOLCADURA       | ██                                         493
OTRO TIPO       | █                                          289
CAIDA           |                                            153
INCENDIO        |                                              2


### Interpretación inicial

En esta capa, **la tarde (12–17) concentra 21.493 registros (37,12%)**, el mayor porcentaje entre los cuatro tramos; la hora exacta con más registros es **18:00**, con 4.497 (7,77%). En la tarde, las causas más frecuentes son distracción del conductor (8.443) e imprudencia del conductor (4.755); el tipo más frecuente es colisión (13.020).

Esto describe la distribución registrada. No demuestra que la hora cause los siniestros ni que la tarde sea más riesgosa: faltan denominadores de circulación y la capa tiene cobertura geocodificada limitada.


## 6. Limitaciones del conjunto

- El análisis usa la capa geocodificada filtrada para 2021, no una base completa certificada de todos los siniestros registrados en Chile. CONASET señala que las capas geocodificadas solo incluyen los casos que se pudieron ubicar; para cálculos sobre la base original recomienda solicitar esos registros.
- La capa seleccionada tiene 57.902 filas y `ZONA` declara 1.363 como rurales; se informa esa diferencia y no se reclasifican filas.
- Un único año permite estudiar distribución temporal dentro de 2021, no tendencias plurianuales.
- Sin flujo vehicular o exposición por hora, los resultados son conteos, no tasas de riesgo. Las causas registradas describen asociación, no causalidad.
- El elemento de ArcGIS no declara una licencia explícita en los metadatos consultados; se cita a CONASET y se deja esa condición pendiente de verificación antes de redistribuir.
- La versión limpia omite direcciones, coordenadas y campos auxiliares para mantener solo lo necesario para la pregunta; la extracción original adjunta conserva los 36 campos.


In [19]:
SALIDA = Path('mi_conjunto_limpio_2021.csv')
df_limpio.to_csv(SALIDA, index=False, encoding='utf-8-sig')
print('Conjunto limpio guardado en:', SALIDA.name)
print('Dimensiones:', df_limpio.shape)
print('Tamaño del archivo (MB):', round(SALIDA.stat().st_size / 1024**2, 2))


Conjunto limpio guardado en: mi_conjunto_limpio_2021.csv
Dimensiones: (57902, 22)
Tamaño del archivo (MB): 12.14
